In [12]:
import os
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv  
from langchain_core.messages import BaseMessage # The foundational class for all message types in LangGraph
from langchain_core.messages import SystemMessage # Message for providing instructions to the LLM
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode


load_dotenv()

google_api_key = os.environ["GOOGLE_API_KEY"]

free_model_names = [
    # Current stable Gemini models
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-flash-lite",

    # Preview models
    "gemini-3-flash-preview",

    # Legacy models - restricted/deprecated
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
    "gemini-2.5-pro",
]

models = [
    ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=google_api_key,
        max_retries=0,
        timeout=30,
    )
    for model_name in free_model_names
]
llm = models[0].with_fallbacks(models[1:])

In [13]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details"""
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [14]:
model_with_structure = llm.with_structured_output(Movie)
model_with_structure

RunnableWithFallbacks(runnable=_ChatModelBinding(bound=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-google-genai': '4.5.0'}}, profile={'name': 'Gemini 3.8 Flash', 'release_date': '2026-09-02', 'last_updated': '2026-09-02', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, google_api_key=SecretStr('**********'), model='gemini-3.8-flash', temperature=None, max_retries=0, timeout=30.0, client=<google.genai.client.Client object at 0x10a3e5370>, default_metadata=(), model_kwargs={}), kwargs={'response_mime_type':

In [15]:
llm.invoke("tell me about the movie intersteller in 5 lines")

AIMessage(content=[{'type': 'text', 'text': 'Here is a 5-line summary of *Interstellar*:\n\n1. Directed by Christopher Nolan, *Interstellar* is a sci-fi epic set in a future where a dying Earth forces humanity to look to the stars for survival.\n2. Former pilot Cooper leaves his family behind to lead a secret NASA mission through a wormhole near Saturn in search of a new home planet.\n3. The crew faces mind-bending space travel, exploring foreign worlds affected by extreme gravity and severe time dilation.\n4. Beyond its complex physics, the heart of the story centers on the powerful, emotional bond between Cooper and his daughter across space and time.\n5. Highlighted by Hans Zimmer’s iconic score and visual effects, it is a breathtaking masterpiece about love, sacrifice, and human endurance.', 'extras': {'signature': 'EvgRCvURAWkUfRNCK9LyNFpsOA4T/q8hIDf0/kJ2vYPyHvRWQMccCHGCds3Wi1k8BDjOtOyRMpl6R8sxAo87kukGajwyl1DHDbryn7TjI9JoaPId0kKCXPUx/Gb3kOeH4Z3uebTdX7l38yasITEogHViPljXSG18P5ESuODX

In [16]:
response = model_with_structure.invoke("tell me about the movie intersteller in 5 lines")
response

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.7)

In [17]:
model_with_structure1 = llm.with_structured_output(Movie,include_raw=True)

In [18]:
response = model_with_structure1.invoke("tell me about the movie intersteller in 5 lines")
response

{'raw': AIMessage(content=[{'type': 'text', 'text': '{"title":"Interstellar","year":2014,"director":"Christopher Nolan","rating":8.7}', 'extras': {'signature': 'EpMHCpAHAWkUfRNMSX+Z8ZBL+cmOFx4OL8a3cIwlGnB1BLLMHZJ6h+Yc87WCpWSK1BgKme44DyOpjgjfnjHJjzSnInGOivC2Nknts+vcKCgp4LP4Ffl/unJOSRYhcwJ49Knw2ng+8uyJe4P068IH4sCHIQn/UxoN8WczUFHhNHeEZrEhCNx1vJqNVxy1Q4Ry9/2oe/d2bC/LlCBlCNXn7ja/yW6TmaSU2H4yG8NgwpDvd6MnZFwQEgzTvNiWvAsyTl4xqUwPlwsIFPQg4BB32+oas+tqH1LC+06dgfw9S1JJjndRoOI/4S/kEUj1xFNRx5Ph14582t8A0MpKC1oeymCQ+AJ/RwjweDFKBxVlihMnjiiNL+FAxveY7oF9IdSwPn4LVlyv6CX9EXIIAtx/D/1h9CDIdgWvpVS99WQkDq8qcPG/W0wd4urpNauXOY1Uzl1DRD74B3VwYRZWKKjre1Gt/bxcB7Je4wpKA7Z21fELXVGSuQVEfjpWlrAN3YT1o1Lk0Kl7a5jnHqn062OnQYcl6EVpawqWHMQVtJPhAgZyD8LAQYfOUze76vKJnu/0/oHc3SP7vWG1HiXLFfKyD81kheL0EpNtx8jOrw8BmyU8pZHX9u3IWwMk3NKXg44qVPHVIB3/LbiBTWwlfhBAtGQjXYHpLatVO8XDciiyS9VkOEbZafru8za5wlIqeFDe4RsSBViyorc1jsmreqSiDFFNL18QfvX/9uvWD414JO9vfQRA834bf4PChMe+8PZg++2GrrOGQ7/Ii5HAGZsof4VYZRTEJkgVsus3XT37YjLZf1rSEReU4/Z65LWaXDr7FvnvajW

In [19]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure2 = llm.with_structured_output(MovieDetails)

response = model_with_structure2.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160.0)

# Typedict

In [21]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = llm.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")